In [ ]:
import numpy as np
import mne

from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import CSP
from sklearn.preprocessing import LabelEncoder

def load_subject(subject_id, data_path):

    filename = f"A{subject_id:02d}T.gdf"
    filepath = f"{data_path}/{filename}"

    print(f"Loading Subject A{subject_id:02d}...")

    # Load GDF
    raw = mne.io.read_raw_gdf(
        filepath,
        preload=True,
        verbose=False
    )

    # Set EOG channels
    eog_channels = ["EOG-left", "EOG-central", "EOG-right"]

    for ch in eog_channels:
        if ch in raw.ch_names:
            raw.set_channel_types({ch: "eog"})

    # Keep EEG only
    raw.pick_types(eeg=True)

    # Bandpass filter
    raw.filter(
        8.,
        30.,
        fir_design="firwin",
        verbose=False
    )

    # Get events
    events, event_id = mne.events_from_annotations(
        raw,
        verbose=False
    )

    # Motor imagery events
    motor_event_ids = {
        "Left_hand": event_id["769"],
        "Right_hand": event_id["770"],
        "feet": event_id["771"],
        "tongue": event_id["772"]
    }

    # Create epochs
    epochs = mne.Epochs(
        raw,
        events,
        event_id=motor_event_ids,
        tmin=0,
        tmax=4,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()
    y = epochs.events[:, -1]
    labelencoder = LabelEncoder()
    y = labelencoder.fit_transform(y)


    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Classes:", np.unique(y))

    return X, y

data_path = "data/BCICIV_2a_gdf"
X_A01, y_A01 = load_subject(1, data_path)

# CSP + LDA

In [ ]:
def run_csp_lda(X, y, random_state=42):

    # -------------------------
    # Train / Test split
    # -------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=random_state
    )

    print("Train:", X_train.shape)
    print("Test :", X_test.shape)

    # -------------------------
    # CSP
    # -------------------------
    csp = CSP(
        n_components=10,
        reg=None,
        log=True,
        norm_trace=False
    )

    X_train_csp = csp.fit_transform(
        X_train,
        y_train
    )

    X_test_csp = csp.transform(
        X_test
    )

    # -------------------------
    # LDA
    # -------------------------
    lda = LinearDiscriminantAnalysis()

    lda.fit(
        X_train_csp,
        y_train
    )

    y_pred = lda.predict(
        X_test_csp
    )

    accuracy = np.mean(
        y_pred == y_test
    )

    return accuracy, y_test, y_pred

accuracy_A01, y_test_A01, y_pred_A01 = run_csp_lda(
    X_A01,
    y_A01
)

print(f"A01 CSP + LDA Accuracy: {accuracy_A01:.3f}")

In [ ]:
csp_results = {}

for subject in range(1, 10):

    print("\n" + "="*50)
    print(f"Subject A{subject:02d}")
    print("="*50)

    X, y = load_subject(
        subject,
        data_path
    )

    acc, y_test, y_pred = run_csp_lda(
        X,
        y
    )

    csp_results[f"A{subject:02d}"] = acc

    print(
        f"A{subject:02d} CSP + LDA: "
        f"{acc*100:.2f}%"
    )

print("\n===== CSP + LDA Results =====")

for subject, acc in csp_results.items():
    print(f"{subject}: {acc*100:.2f}%")

mean_acc = np.mean(
    list(csp_results.values())
)

std_acc = np.std(
    list(csp_results.values()),
    ddof=1
)

print(
    f"\nMean ± SD: "
    f"{mean_acc*100:.2f}% ± {std_acc*100:.2f}%"
)

# EEG

In [ ]:
import copy
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split


class EEGNet(nn.Module):

    def __init__(self,n_classes=4):
        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=(1, 64),
            padding=(8, 32),
            bias=False,
            ), nn.BatchNorm2d(8)
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(
                in_channels=8,
                out_channels=16,
                kernel_size=(22, 1),
                groups=8,
                bias=False,
            ),
            nn.BatchNorm2d(16),
            nn.ELU()
        )

        self.pool1 = nn.AvgPool2d(
            kernel_size=(1, 4),
        )

        self.dropout1 = nn.Dropout(p= 0.25)

        self.separable = nn.Sequential(
          nn.Conv2d(
            in_channels=16,
            out_channels=16,
            kernel_size=(1, 16),
            padding=(0, 8),
            groups=16,
            bias=False
        ),

        nn.Conv2d(
            in_channels=16,
            out_channels=16,
            kernel_size=(1, 1),
            bias=False
        ),

        nn.BatchNorm2d(16),
        nn.ELU()
        )


        self.pool2 = nn.AvgPool2d(
            kernel_size=(1, 8)
        )

        self.dropout2 = nn.Dropout(
            p=0.25
        )


        self.classifier = nn.LazyLinear(n_classes)


    def forward(self, x):
        x = self.temporal(x)


        x = self.spatial(x)


        x = self.pool1(x)
        x = self.dropout1(x)

        x = self.separable(x)

        x = self.pool2(x)
        x = self.dropout2(x)

        x = torch.flatten(x, start_dim=1)

        x = self.classifier(x)

        return x


def run_eegnet(
    X,
    y,
    random_state=42,
    num_epochs=50,
    batch_size=16,
    learning_rate=0.001
):

    # =====================================================
    # 1. Train / Test split
    # =====================================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=random_state
    )


    # =====================================================
    # 2. Train / Validation split
    # =====================================================

    X_train, X_val, y_train, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=random_state
    )


    # =====================================================
    # 3. Create DataLoaders
    # =====================================================

    train_dataset = TensorDataset(
        X_train,
        y_train
    )

    val_dataset = TensorDataset(
        X_val,
        y_val
    )

    test_dataset = TensorDataset(
        X_test,
        y_test
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )


    # =====================================================
    # 4. NEW EEGNet for this subject
    # =====================================================

    model = EEGNet(n_classes=4)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )


    # =====================================================
    # 5. Accuracy function
    # =====================================================

    def evaluate(loader):

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for X_batch, y_batch in loader:

                outputs = model(X_batch)

                predictions = outputs.argmax(dim=1)

                correct += (
                    predictions == y_batch
                ).sum().item()

                total += y_batch.size(0)

        return correct / total


    # =====================================================
    # 6. Training
    # =====================================================

    best_val_acc = 0.0
    best_model_state = None

    train_losses = []
    val_accs = []

    for epoch in range(num_epochs):

        model.train()

        running_loss = 0.0

        for X_batch, y_batch in train_loader:

            outputs = model(X_batch)

            loss = criterion(
                outputs,
                y_batch
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = (
            running_loss /
            len(train_loader)
        )

        val_acc = evaluate(val_loader)

        train_losses.append(epoch_loss)
        val_accs.append(val_acc)


        # Save best validation model
        if val_acc > best_val_acc:

            best_val_acc = val_acc

            best_model_state = copy.deepcopy(
                model.state_dict()
            )


        print(
            f"Epoch [{epoch+1:2d}/{num_epochs}] "
            f"| Loss: {epoch_loss:.4f} "
            f"| Val Acc: {val_acc:.3f}"
        )


    # =====================================================
    # 7. Load best model
    # =====================================================

    model.load_state_dict(
        best_model_state
    )


    # =====================================================
    # 8. Final test evaluation
    # =====================================================

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            outputs = model(X_batch)

            predictions = outputs.argmax(dim=1)

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                y_batch.cpu().numpy()
            )


    all_predictions = np.array(
        all_predictions
    )

    all_labels = np.array(
        all_labels
    )

    test_accuracy = np.mean(
        all_predictions == all_labels
    )


    return {
        "test_accuracy": test_accuracy,
        "best_val_accuracy": best_val_acc,
        "y_test": all_labels,
        "y_pred": all_predictions,
        "train_losses": train_losses,
        "val_accs": val_accs,
        "model": model
    }

In [ ]:
X_A01_torch = torch.tensor(
    X_A01,
    dtype=torch.float32
).unsqueeze(1)

y_A01_torch = torch.tensor(
    y_A01,
    dtype=torch.long
)

result_A01 = run_eegnet(
    X_A01_torch,
    y_A01_torch
)

print(
    "A01 Best Validation Accuracy:",
    result_A01["best_val_accuracy"]
)

print(
    "A01 Test Accuracy:",
    result_A01["test_accuracy"]
)

In [ ]:
eegnet_results = {}

for subject_id in range(2, 10):
    print(f"\n{'='*50}")
    print(f"Subject A{subject_id:02d}")
    print(f"{'='*50}")

    # Load subject data
    X, y = load_subject(subject_id, data_path)

    # Convert to PyTorch tensors
    X_torch = torch.tensor(
        X, dtype=torch.float32
    ).unsqueeze(1)

    y_torch = torch.tensor(
        y, dtype=torch.long
    )

    # Train and evaluate EEGNet
    result = run_eegnet(
        X_torch,
        y_torch,
        random_state=42,
        num_epochs=50,
        batch_size=16,
        learning_rate=0.001
    )

    eegnet_results[f"A{subject_id:02d}"] = result

    print(
        f"A{subject_id:02d} Test Accuracy: "
        f"{result['test_accuracy']:.4f}"
    )

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

class_names = [
    "Left hand",
    "Right hand",
    "Feet",
    "Tongue"
]

for subject in eegnet_results:

    result = eegnet_results[subject]

    cm = confusion_matrix(
        result["y_test"],
        result["y_pred"],
        labels=[0, 1, 2, 3]
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    fig, ax = plt.subplots(figsize=(6, 5))

    disp.plot(
        ax=ax,
        values_format="d",
        cmap=None
    )

    ax.set_title(f"EEGNet Confusion Matrix — {subject}")
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(15, 11))

for ax, subject in zip(axes.flat, eegnet_results):

    result = eegnet_results[subject]

    epochs = range(1, len(result["train_losses"]) + 1)

    ax2 = ax.twinx()

    ax.plot(
        epochs,
        result["train_losses"],
        label="Train Loss"
    )

    ax2.plot(
        epochs,
        result["val_accs"],
        label="Val Accuracy"
    )

    ax.set_title(subject)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train Loss")
    ax2.set_ylabel("Val Accuracy")
    ax2.set_ylim(0, 1)

plt.suptitle("EEGNet Training and Validation Curves", fontsize=16)
plt.tight_layout()
plt.show()